# UTS Data Science - Dokumentasi Praktikum
## Pertemuan 6: Persiapan Data (Data Preprocessing dengan Scikit-Learn)

* **Nama Lengkap:** Martin Hotmatua Siregar
* **NIM:** 240401020111
* **Kelas:** IF403
* **Program Studi:** PJJ Informatika
* **Instansi:** Universitas Siber Asia

## Langkah 1 s.d 3: Memuat Dataset, Memisahkan Fitur-Target, & Train-Test Split
Kita akan memuat dataset Titanic langsung dari repositori publik, memisahkan kolom 'Survived' sebagai target prediksi ($y$), dan membagi data menjadi 80% untuk pelatihan (*train set*) serta 20% untuk pengujian (*test set*) menggunakan metode acak terstratifikasi (*Stratified Split*) agar proporsi kelas tetap seimbang.

In [1]:
# Mengimpor library dasar dan fungsi pemisah data dari scikit-learn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# Langkah 1: Memuat dataset Titanic dari URL resmi
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

# Memilih kolom-kolom penting yang relevan untuk pemodelan agar ringkas
kolom_pilihan = ['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare']
df_titanic = df[kolom_pilihan].copy()

# Langkah 2: Memisahkan Fitur (X) dan Target/Label (y)
X = df_titanic.drop('Survived', axis=1)
y = df_titanic['Survived']

# Langkah 3: Melakukan Train-Test Split (80% Train, 20% Test)
# Menggunakan parameter stratify=y agar persentase keselamatan seimbang di kedua subset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("=== VERIFIKASI PEMBAGIAN DATA ===")
print(f"Ukuran X_train (Data Latih) : {X_train.shape}")
print(f"Ukuran X_test (Data Uji)    : {X_test.shape}")
print(f"Ukuran y_train (Label Latih): {y_train.shape}")
print(f"Ukuran y_test (Label Uji)   : {y_test.shape}\n")

print("Proporsi kelas target pada Data Latih:")
print(y_train.value_counts(normalize=True).round(3))

=== VERIFIKASI PEMBAGIAN DATA ===
Ukuran X_train (Data Latih) : (712, 6)
Ukuran X_test (Data Uji)    : (179, 6)
Ukuran y_train (Label Latih): (712,)
Ukuran y_test (Label Uji)   : (179,)

Proporsi kelas target pada Data Latih:
Survived
0    0.617
1    0.383
Name: proportion, dtype: float64


**Analisis Temuan Langkah 1 s.d 3:**
Proses pemisahan data berhasil membagi 891 baris data asli menjadi 712 baris untuk porsi data latih (*train*) dan 179 baris untuk data uji (*test*). Berkat penerapan parameter `stratify=y`, proporsi penumpang yang tidak selamat (0) dan selamat (1) pada data latih terkunci seimbang di angka 61.7% berbanding 38.3%, persis menyerupai distribusi pada data uji dan dataset asli. Hal ini sangat penting untuk mencegah model bias saat belajar.

## Langkah 4 s.d 5: Imputasi Data Kosong, Encoding Kategorikal, & Feature Scaling
Pada tahap krusial ini, kita akan melakukan transformasi data secara terpisah pada Fitur Latih (*Train*) dan Fitur Uji (*Test*). Kita tambal nilai kosong pada variabel 'Age' dengan nilai median, ubah fitur string 'Sex' menjadi bentuk matriks biner (angka 0 dan 1) lewat One-Hot Encoding, serta standarkan rentang nilai 'Age' dan 'Fare' menggunakan teknik Z-Score standardisasi (`StandardScaler`).

In [2]:
# Membuat salinan data untuk proses transformasi agar data asli tidak rusak
X_train_clean = X_train.copy()
X_test_clean = X_test.copy()

# --- LANGKAH 4: HANDLING MISSING VALUES (IMPUTASI MEDIAN) ---
imputer_age = SimpleImputer(strategy='median')
# Menghitung median hanya dari data latih (fit) lalu menerapkannya (transform) ke train dan test
X_train_clean['Age'] = imputer_age.fit_transform(X_train_clean[['Age']])
X_test_clean['Age'] = imputer_age.transform(X_test_clean[['Age']])


# --- LANGKAH 5: ENCODING KATEGORIKAL (ONE-HOT ENCODING) ---
encoder_sex = OneHotEncoder(sparse_output=False, drop='first') # drop='first' untuk menghindari jebakan multikolinieritas
# Mengonversi kolom teks 'Sex' menjadi fitur numerik biner (0 atau 1)
sex_train_encoded = encoder_sex.fit_transform(X_train_clean[['Sex']])
sex_test_encoded = encoder_sex.transform(X_test_clean[['Sex']])

# Mengubah hasil encoding menjadi DataFrame dengan nama kolom baru yang representatif
kolom_sex_baru = encoder_sex.get_feature_names_out(['Sex'])
df_sex_train = pd.DataFrame(sex_train_encoded, columns=kolom_sex_baru, index=X_train_clean.index)
df_sex_test = pd.DataFrame(sex_test_encoded, columns=kolom_sex_baru, index=X_test_clean.index)

# Menggabungkan kembali kolom hasil encoding dan membuang kolom teks 'Sex' lama
X_train_clean = pd.concat([X_train_clean.drop('Sex', axis=1), df_sex_train], axis=1)
X_test_clean = pd.concat([X_test_clean.drop('Sex', axis=1), df_sex_test], axis=1)


# --- LANGKAH 6: FEATURE SCALING (STANDARD SCALER) ---
# Menyetarakan skala kolom numerik kontinyu ('Age' dan 'Fare') agar model berbasis jarak tidak bias
fitur_numerik = ['Age', 'Fare']
scaler_dataset = StandardScaler()

X_train_clean[fitur_numerik] = scaler_dataset.fit_transform(X_train_clean[fitur_numerik])
X_test_clean[fitur_numerik] = scaler_dataset.transform(X_test_clean[fitur_numerik])

print("=== PIPELINE PREPROCESSING BERHASIL ===")
print("\nContoh 3 baris teratas hasil akhir Preprocessing Data Latih (X_train_clean):")
display(X_train_clean.head(3).round(3))

=== PIPELINE PREPROCESSING BERHASIL ===

Contoh 3 baris teratas hasil akhir Preprocessing Data Latih (X_train_clean):


,Pclass,Age,SibSp,Parch,Fare,Sex_male
692,3,-0.081,0,0,0.514,1.0
481,2,-0.081,0,0,-0.663,1.0
527,1,-0.081,0,0,3.955,1.0


**Analisis Temuan Langkah 4 s.d 5:**
Melalui transformasi di atas, kolom teks `Sex` kini telah berubah menjadi kolom numerik `Sex_female` (atau `Sex_male` bernilai 0 dan 1) sehingga dapat dicerna oleh algoritma komputer. Kolom `Age` dan `Fare` yang semula memiliki satuan rentang nilai sangat timpang (Usia di skala puluhan tahun, sedangkan Fare di skala ratusan dolar) sekarang sudah disetarakan skalanya menjadi distribusi seragam dengan nilai rata-rata (*mean*) mendekati 0 dan nilai standar deviasi bernilai 1.

## Kesimpulan & Refleksi Pembelajaran (Sesi 6)

### 1. Apa yang Dipelajari?
Pada pertemuan keenam ini, saya mempelajari seni mempersiapkan data (*data preprocessing*) sebelum dimasukkan ke dalam tahap pemodelan algoritma kecerdasan buatan. Saya mempraktikkan alur kerja menggunakan pustaka Scikit-Learn untuk menambal data kosong secara otomatis dengan `SimpleImputer`, mengubah data teks kategorikal menggunakan metode `OneHotEncoder`, serta menyetarakan timpangnya rentang ukuran data kontinyu menggunakan teknik `StandardScaler`.

### 2. Temuan Utama
* Aturan urutan kerja dalam pengerjaan data sains itu bersifat mutlak: Pembagian data (*Train-Test Split*) harus dieksekusi terlebih dahulu sebelum melakukan kalkulasi nilai median (imputasi) atau mean/std (scaling). Hal ini wajib dilakukan demi mencegah terjadinya kebocoran data (*data leakage*) yang bisa mengakibatkan performa penilaian model menjadi menipu atau terlalu optimis semu.
* Sebagian besar model matematika Machine Learning (seperti regresi, KNN, atau SVM) sangat sensitif terhadap skala data. Penyamaan skala lewat StandardScaler terbukti mampu meredam dominasi fitur yang nilai angkanya besar terhadap fitur yang angkanya kecil.

### 3. Keterbatasan & Pertanyaan yang Muncul
* **Keterbatasan:** Teknik One-Hot Encoding yang diaplikasikan pada praktikum ini akan melipatgandakan jumlah kolom baru secara drastis jika variabel teks kategorikalnya memiliki ribuan nilai unik yang berbeda (kardinalitas tinggi).
* **Pertanyaan:** Jika di dalam dataset riil kita menemui kolom teks kategorikal bertingkat yang memiliki urutan jelas (seperti tingkat pendidikan: SD, SMP, SMA, S1), apakah metode One-Hot Encoding ini tetap menjadi pilihan terbaik, atau ada alternatif encoding lain yang lebih efisien untuk mempertahankan urutan tersebut?